# 00 · Geração da base de relatos de obstáculos (dados sintéticos)

**Projeto:** Waze da Acessibilidade · Global Solution 2026/2 · CitizenTech
**Disciplina:** Artificial Intelligence & Chatbot

Este notebook **gera** o arquivo `dados/relatos_obstaculos.csv`, usado pelo **Modelo 2 (zonas críticas)**.

### Por que a base é sintética?
O app ainda não está em produção, então ainda não existem relatos reais de usuários.
Para treinar o modelo de agrupamento, **simulamos** como seriam esses relatos em uma região real de São Paulo.

### Como a simulação foi feita
1. Escolhemos **7 bairros reais** da região central e oeste de São Paulo (Sé, República, Liberdade, Bela Vista, Consolação, Pinheiros e Vila Mariana).
2. Definimos **6 focos de problemas** em locais reais, conhecidos por calçadas difíceis (por exemplo, a Rua 25 de Março, com muito comércio de rua, e a Escadaria do Bixiga).
3. Cada foco recebe uma quantidade de relatos espalhados em volta do ponto central (distribuição normal com desvio de ~110 a 190 metros, o tamanho de poucos quarteirões).
4. Cada foco tem tipos de obstáculo mais comuns (ex.: Bixiga → mais escadas sem rampa; 25 de Março → mais obstruções).
5. **15% dos relatos** são espalhados aleatoriamente pela região, representando obstáculos isolados.
6. O bairro de cada relato é o bairro cujo centro está mais perto.
7. De propósito, inserimos **alguns problemas comuns em dados reais** (relatos duplicados e bairros vazios), para o notebook do Modelo 2 mostrar o tratamento.

Usamos `random_state = 42` (semente fixa), então quem rodar este notebook obtém **exatamente o mesmo arquivo**.

In [1]:
import numpy as np
import pandas as pd

SEMENTE = 42
rng = np.random.default_rng(SEMENTE)

## 1. Bairros e focos de problemas
As coordenadas são aproximadas e foram conferidas no mapa. Cada foco tem:
- `qtd`: quantos relatos gerar;
- `desvio_m`: quanto os relatos se espalham ao redor do centro (em metros);
- `tipos`: a probabilidade de cada tipo de obstáculo naquele foco.

In [2]:
TIPOS = ["buraco", "calcada_irregular", "escada_sem_rampa", "obstrucao_calcada"]

# centro aproximado de cada bairro (latitude, longitude)
BAIRROS = {
    "Sé":          (-23.5505, -46.6333),
    "República":   (-23.5440, -46.6430),
    "Liberdade":   (-23.5600, -46.6350),
    "Bela Vista":  (-23.5610, -46.6480),
    "Consolação":  (-23.5520, -46.6600),
    "Pinheiros":   (-23.5650, -46.6900),
    "Vila Mariana": (-23.5880, -46.6380),
}

# focos de problemas:      lat       lon     qtd desvio  [buraco, calcada, escada, obstrucao]
FOCOS = {
    "Rua 25 de Março":     (-23.5430, -46.6305, 230, 120, [0.15, 0.25, 0.05, 0.55]),
    "Praça da Sé":         (-23.5503, -46.6340, 190, 140, [0.20, 0.45, 0.10, 0.25]),
    "Escadaria do Bixiga": (-23.5585, -46.6450, 160, 110, [0.10, 0.30, 0.50, 0.10]),
    "Largo da Batata":     (-23.5670, -46.6935, 150, 170, [0.45, 0.25, 0.05, 0.25]),
    "Praça da Liberdade":  (-23.5555, -46.6355, 120, 110, [0.15, 0.35, 0.30, 0.20]),
    "Metrô Vila Mariana":  (-23.5890, -46.6345, 100, 190, [0.25, 0.55, 0.05, 0.15]),
}

# área total da simulação (retângulo que cobre os 7 bairros)
LAT_MIN, LAT_MAX = -23.595, -23.538
LON_MIN, LON_MAX = -46.700, -46.625

## 2. Gerando os relatos
Para converter metros em graus usamos uma aproximação simples:
1 grau de latitude ≈ 111 km e 1 grau de longitude ≈ 111 km × cos(latitude) ≈ 102 km em São Paulo.

In [3]:
METROS_POR_GRAU_LAT = 111_000
METROS_POR_GRAU_LON = 111_000 * np.cos(np.radians(-23.56))

linhas = []
for nome, (lat, lon, qtd, desvio_m, probs) in FOCOS.items():
    lats = rng.normal(lat, desvio_m / METROS_POR_GRAU_LAT, qtd)
    lons = rng.normal(lon, desvio_m / METROS_POR_GRAU_LON, qtd)
    tipos = rng.choice(TIPOS, size=qtd, p=probs)
    for la, lo, t in zip(lats, lons, tipos):
        linhas.append((la, lo, t))

# 15% de relatos isolados, espalhados pela área toda
qtd_focos = len(linhas)
qtd_isolados = int(qtd_focos * 0.15 / 0.85)
lats = rng.uniform(LAT_MIN, LAT_MAX, qtd_isolados)
lons = rng.uniform(LON_MIN, LON_MAX, qtd_isolados)
tipos = rng.choice(TIPOS, size=qtd_isolados, p=[0.30, 0.35, 0.10, 0.25])
linhas += list(zip(lats, lons, tipos))

df = pd.DataFrame(linhas, columns=["latitude", "longitude", "tipo_obstaculo"])
print("Relatos em focos:", qtd_focos, "| Relatos isolados:", qtd_isolados, "| Total:", len(df))

Relatos em focos: 950 | Relatos isolados: 167 | Total: 1117


In [4]:
# bairro = bairro com o centro mais próximo
nomes_bairros = list(BAIRROS.keys())
centros = np.array(list(BAIRROS.values()))

def bairro_mais_proximo(lat, lon):
    dist = np.sqrt(((centros[:, 0] - lat) * METROS_POR_GRAU_LAT) ** 2 +
                   ((centros[:, 1] - lon) * METROS_POR_GRAU_LON) ** 2)
    return nomes_bairros[int(np.argmin(dist))]

df["bairro"] = [bairro_mais_proximo(la, lo) for la, lo in zip(df["latitude"], df["longitude"])]

# data do relato: entre 01/01/2026 e 31/10/2026
inicio = pd.Timestamp("2026-01-01")
dias = rng.integers(0, 304, len(df))
df["data_relato"] = (inicio + pd.to_timedelta(dias, unit="D")).strftime("%Y-%m-%d")

df["latitude"] = df["latitude"].round(6)
df["longitude"] = df["longitude"].round(6)
df.head()

,latitude,longitude,tipo_obstaculo,bairro,data_relato
0,-23.542671,-46.632226,escada_sem_rampa,Sé,2026-06-08
1,-23.544124,-46.631197,escada_sem_rampa,Sé,2026-05-14
2,-23.542189,-46.630128,calcada_irregular,Sé,2026-02-13
3,-23.541983,-46.629078,obstrucao_calcada,Sé,2026-03-23
4,-23.545109,-46.631360,calcada_irregular,Sé,2026-09-25


## 3. Inserindo problemas reais de qualidade de dados
Em um app de verdade, é comum:
- o mesmo usuário enviar o **mesmo relato duas vezes** (duplicados);
- o app **não conseguir identificar o bairro** (campo vazio).

Inserimos 15 duplicados e 20 bairros vazios. O notebook do Modelo 2 vai detectar e tratar esses casos.

In [5]:
duplicados = df.sample(15, random_state=SEMENTE)
df = pd.concat([df, duplicados], ignore_index=True)

sem_bairro = df.sample(20, random_state=SEMENTE + 1).index
df.loc[sem_bairro, "bairro"] = np.nan

# embaralha as linhas e cria o id
df = df.sample(frac=1, random_state=SEMENTE).reset_index(drop=True)
df.insert(0, "id", range(1, len(df) + 1))
df = df[["id", "latitude", "longitude", "tipo_obstaculo", "data_relato", "bairro"]]

df.to_csv("../dados/relatos_obstaculos.csv", index=False, encoding="utf-8")
print("Arquivo salvo com", len(df), "linhas")
df.head(10)

Arquivo salvo com 1132 linhas


,id,latitude,longitude,tipo_obstaculo,data_relato,bairro
0,1,-23.556590,-46.634622,buraco,2026-10-20,Liberdade
1,2,-23.588687,-46.633594,calcada_irregular,2026-10-27,Vila Mariana
2,3,-23.551175,-46.635895,calcada_irregular,2026-01-14,Sé
3,4,-23.557662,-46.645123,escada_sem_rampa,2026-03-26,Bela Vista
4,5,-23.555621,-46.633408,calcada_irregular,2026-10-28,Liberdade
5,6,-23.544867,-46.629782,calcada_irregular,2026-01-13,Sé
6,7,-23.558751,-46.644927,escada_sem_rampa,2026-05-11,Bela Vista
7,8,-23.543297,-46.630189,calcada_irregular,2026-05-13,Sé
8,9,-23.542636,-46.643907,obstrucao_calcada,2026-07-04,República
9,10,-23.555256,-46.636256,calcada_irregular,2026-06-15,Liberdade


In [6]:
# conferência rápida
print(df["tipo_obstaculo"].value_counts(), "\n")
print(df["bairro"].value_counts(dropna=False))

tipo_obstaculo
calcada_irregular    368
obstrucao_calcada    313
buraco               278
escada_sem_rampa     173
Name: count, dtype: int64 

bairro
Sé              466
Pinheiros       221
Bela Vista      173
Vila Mariana    131
Liberdade        90
NaN              20
Consolação       17
República        14
Name: count, dtype: int64


## Resumo
- Base **sintética**, gerada com semente fixa (`42`), salva em `dados/relatos_obstaculos.csv`.
- Colunas: `id`, `latitude`, `longitude`, `tipo_obstaculo`, `data_relato`, `bairro`.
- Os 4 tipos de obstáculo são **os mesmos do Modelo 1**: em produção, cada foto autenticada pelo Modelo 1 vira uma linha desta tabela.
- **Limitação:** como os dados são simulados, as zonas encontradas pelo Modelo 2 refletem os focos que nós definimos. Com relatos reais, o mesmo código encontraria as zonas reais da cidade.